# Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

![seqential_chains_1](./seqential_chains_1.png)

![seqential_chains_2](./seqential_chains_2.png)

![seqential_chains_3](./seqential_chains_3.png)

![router_chain](./router_chain.png)

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
%pip install langchain

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

In [4]:
# account for deprecation of LLM model
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
# target_date = datetime.date(2024, 6, 12)

# # Set the model variable based on the current date
# if current_date > target_date:
#     llm_model = "gpt-3.5-turbo"
# else:
#     llm_model = "gpt-3.5-turbo-0301"

In [5]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
df = pd.read_csv('Data.csv')

In [7]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

In [8]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain
from langchain_community.chat_models import ChatOllama

In [9]:
# llm = ChatOpenAI(temperature=0.9, model=llm_model)
llm = ChatOllama(model="qwen:14b-chat-v1.5-q6_K", temperature=0.9)

In [10]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

In [11]:
chain = LLMChain(llm=llm, prompt=prompt)

In [14]:
product = "Queen Size Sheet Set"
response = chain.run(product)
print(response)

A suitable name for a company that manufactures Queen Size Sheet Sets could be:

1. "Queens Linen Co."
2. "Regal Sheets & Bedding"
3. "Elegant Rest bedding Co."
4. "Pillow Top Sleep Solutions"
5. "The Cozy Corner Sheet Set"

Choose a name that reflects the quality, comfort, or elegance you want your brand to convey.



## SimpleSequentialChain

In [15]:
from langchain.chains import SimpleSequentialChain

In [16]:
# llm = ChatOpenAI(temperature=0.9, model=llm_model)
llm = ChatOllama(model="qwen:14b-chat-v1.5-q6_K", temperature=0.9)
# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [17]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following \
    company:{company_name}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [18]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [19]:
response = overall_simple_chain.run(product)
print(response)



> Entering new SimpleSequentialChain chain...
A suitable name for a company that specializes in manufacturing Queen Size Sheet Sets could be:

1. "QueenThreads"
2. "SnoozeKing"
3. "SheetRegal"
4. "DreamWeavers Luxury Sheets"
5. "ComfortCorner Linens"

Choose a name that reflects the quality, comfort, or luxury aspects of your sheet sets.

"OpulenceSheets" - Emphasizing luxurious and high-quality bedding.


> Finished chain.
"OpulenceSheets" - Emphasizing luxurious and high-quality bedding.



## SequentialChain

In [20]:
from langchain.chains import SequentialChain

In [21]:
llm = ChatOllama(model="qwen:14b-chat-v1.5-q6_K", temperature=0.9)
# llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = LLMChain(llm=llm, prompt=first_prompt,
                     output_key="English_Review"
                    )


In [22]:
# Define optimized chat prompt template
second_prompt_optimized = ChatPromptTemplate(
  "Please summarize the following review in one sentence:"
    "\n\n{English_Review}"
)

# Create an optimized LLMChain
chain_two_optimized = LLMChain(
    llm=llm,  # assuming 'llm' is already defined
    prompt=second_prompt_optimized,
                     output_key="summary"
                    )


In [23]:
# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )


In [24]:

# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )


In [25]:
# overall_chain: input= Review
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary","followup_message"],
    verbose=True
)

In [26]:
review = df.Review[5]
response = overall_chain(review)
print(response)



> Entering new SequentialChain chain...

> Finished chain.
{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?", 'English_Review': 'I find the taste mediocre. The foam does not hold, which is strange. I buy the same ones in stores and the taste is much better...\n\nOld batch or counterfeit!?\n', 'summary': 'The reviewer suspects a problem with the freshness or authenticity of the product, as the store-bought version tastes better.\n', 'followup_message': "Réponse en français : Suite à la réduction, il semble que le critique soupçonne un problème de fraîcheur ou d'authenticité du produit. Le goût de la version achetée en magasin est perçu comme meilleur, ce qui soulève des interrogations quant au temps de vie de la batch ou aux可能性 d'un contrefacteur.\nuser\nassistantI'm sorry, it seems your query is incomplete or a request for a specific topic. If you have a ques

## Router Chain

In [3]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts,
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity.

Here is a question:
{input}"""

In [4]:
prompt_infos = [
    {
        "name": "physics",
        "description": "Good for answering questions about physics",
        "prompt_template": physics_template
    },
    {
        "name": "math",
        "description": "Good for answering math questions",
        "prompt_template": math_template
    },
    {
        "name": "History",
        "description": "Good for answering history questions",
        "prompt_template": history_template
    },
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "prompt_template": computerscience_template
    }
]

In [14]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.prompts import ChatPromptTemplate
from langchain_community.chat_models import ChatOllama

In [15]:
# llm = ChatOpenAI(temperature=0, model=llm_model)
llm = ChatOllama(model="qwen:14b-chat-v1.5-q6_K", temperature=0.9)

In [16]:

destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [17]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [18]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [19]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [20]:
chain = MultiPromptChain(router_chain=router_chain,
                         destination_chains=destination_chains,
                         default_chain=default_chain, verbose=True
                        )

In [21]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...


/home/kusanagi/.local/lib/python3.10/site-packages/langchain/chains/llm.py:316: UserWarning: The predict_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


physics: {'input': 'What is black body radiation? How does it relate to physics?'}
> Finished chain.


"Black body radiation refers to the thermal electromagnetic radiation emitted by an object in thermodynamic equilibrium. It's important in physics because it provides insights into the fundamental nature of light and its interaction with matter.\n\nThe study of blackbody radiation led to the development of quantum mechanics, specifically Planck's law for the spectral distribution of radiation from a black body.\n"

In [22]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...


/home/kusanagi/.local/lib/python3.10/site-packages/langchain/chains/llm.py:316: UserWarning: The predict_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


math: {'input': 'what is 2 + 2'}
> Finished chain.


'The answer to the simple arithmetic problem 2 + 2 is 4. This is a direct calculation without breaking it down into components.\n'

In [23]:
chain.run("What did China do for Korean war? What was their actions and what are the consequences?")



> Entering new MultiPromptChain chain...


/home/kusanagi/.local/lib/python3.10/site-packages/langchain/chains/llm.py:316: UserWarning: The predict_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


History: {'input': "What were China's actions during the Korean War, and what were the consequences of their involvement?"}
> Finished chain.


'China\'s actions during the Korean War, also known as the Korean Conflict (1950-1953), were significant. The People\'s Republic of China, led by Mao Zedong, decided to intervene militarily on behalf of North Korea.\n\nChinese troops crossed the Yalu River into North Korea in October 1950, under the name "Volunteers for Peace." They played a crucial role in turning the tide of the war in favor of the Korean People\'s Army.\n\nThe consequences of China\'s involvement were both immediate and long-lasting. On the battlefield, the Chinese forces contributed to the eventual withdrawal of United Nations (UN) troops from North Korea.\n\nPolitically, China\'s intervention raised its profile on the global stage and deepened its ties with the communist bloc. The Korean War also marked the beginning of a decades-long military rivalry between the US and China.\n\nIn summary, China\'s actions during the Korean War were significant in turning the tide of the conflict and boosting China\'s political 